### Environment setup (Kaggle) & Hugging Face Auth

In [ ]:
# Install dependencies

# !pip uninstall -y torchao
!pip install -q -U torchao>=0.16.0
!pip install -q bitsandbytes accelerate
!pip install -q "pytorch-lightning>=1.8.0,<2.0.0"
!pip install -q transformers datasets peft evaluate sacrebleu unbabel-comet huggingface_hub

import os
import re
import unicodedata
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from huggingface_hub import login, notebook_login

# Hugging face auth on Kaggle.
# Add secret HF_TOKEN with a write permission. Add-ons -> Secrets 
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token)
    print("[✓] Authenticated successfully")
except Exception as e:
    print("[!] Secrets not found or unconfigured. Manual login:")
    notebook_login()

# Check GPU available
num_gpus = torch.cuda.device_count()
print(f"[✓] Detected {num_gpus} GPU(s)")

# For languages that were not originally trained with NLLB, we add a similar language code to copy its initaial embeddings.

TARGET_LANG_CODE = "lang_code"             # Can be 'kln_Latn', 'guz_Latn', 'mer_Latn', 'dav_Latn', 'ebu_Latn', 'mas_Latn'. Check out ISO 639-3 Language Code for official language codes
DATASET_FILENAME = "datasetname.csv" 
SIMILAR_LANG_CODE = "lang_code"            # Existing anchor language code('kik_Latn', 'kam_Latn', 'luo_Latn', 'swh_Latn' - these are some of the Kenyan languages originally trained in NLLB)

SRC_COL = "English"
TGT_COL = "Language"                   # Match your CSV target header column name (Maasai, Kisii)

HF_USERNAME = "huggingface-username"
HUB_REPO_NAME = "repository to push trained model"
FULL_REPO_ID = f"{HF_USERNAME}/{HUB_REPO_NAME}"

print(f"[✓] Target Repository ID set to: {FULL_REPO_ID}")

### Data preprocessing

In [ ]:
def load_and_preprocess_pipeline(
    csv_name,
    src_col,
    tgt_col,
    max_src_len=512,
    max_tgt_len=512,
    min_src_len=2,
    min_tgt_len=2,
    drop_on_truncation=True,
    seed=42
):
    path_options = [
        os.path.join("/kaggle/working", csv_name),
        os.path.join("/kaggle/input", csv_name),
        os.path.join("/kaggle/input", csv_name.split('.')[0], csv_name),
        os.path.join(os.getcwd(), csv_name),
    ]

    download_path = None
    for p in path_options:
        if os.path.exists(p):
            download_path = p
            print(f"[*] Found dataset at: {download_path}")
            break
            
    if download_path is None and os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            if csv_name in files:
                download_path = os.path.join(root, csv_name)
                print(f"[*] Found dataset via deep search at: {download_path}")
                break

    if download_path is None:
        raise FileNotFoundError(f"Could not find '{csv_name}': {path_options}.")

    df = pd.read_csv(download_path)
    if src_col not in df.columns or tgt_col not in df.columns:
        print("Column names not found in header. Falling back to default first two columns.")
        df = pd.read_csv(download_path, header=None, names=[src_col, tgt_col])

    df = df.dropna(subset=[src_col, tgt_col]).copy()

    def clean_text(x):
        x = unicodedata.normalize("NFC", str(x))
        x = re.sub(r"\s+", " ", x).strip()
        return x

    df[src_col] = df[src_col].apply(clean_text)
    df[tgt_col] = df[tgt_col].apply(clean_text)

    df = df[
        df[src_col].str.strip().str.lower() != df[tgt_col].str.strip().str.lower()
    ].copy()

    df = df.drop_duplicates(subset=[src_col, tgt_col]).reset_index(drop=True)
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    df = df[[src_col, tgt_col]].reset_index(drop=True)

    raw_dataset = Dataset.from_pandas(df, preserve_index=False)

    train_test = raw_dataset.train_test_split(test_size=0.1, seed=seed, shuffle=True)
    test_valid = train_test["test"].train_test_split(test_size=0.5, seed=seed, shuffle=True)

    final_dataset = DatasetDict({
        "train": train_test["train"],
        "validation": test_valid["train"],
        "test": test_valid["test"],
    })

    print(f"[✓] Data preprocessing complete. Rows kept: {len(df)}")
    print(f"[✓] Split overview:\n{final_dataset}")
    return final_dataset

dataset_package = load_and_preprocess_pipeline(
    DATASET_FILENAME,
    src_col=SRC_COL,
    tgt_col=TGT_COL,
    max_src_len=512,
    max_tgt_len=512,
    min_src_len=2,
    min_tgt_len=2,
    drop_on_truncation=True,
    seed=42
)

In [ ]:
# from numba import cuda
# device = cuda.get_current_device()
# device.reset()

##### Parameter-Efficient Fine-Tuning (PEFT) using Low-Rank Adaptation (LoRA)

In [ ]:
# This notebook used the 2xT4s GPUs provided by Kaggle 

from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

NLLB_MODEL_ID = "facebook/nllb-200-distilled-600M"

print("[*] Loading base tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL_ID, src_lang="eng_Latn")

# Register new language special token
print(f"[*] Injecting unseen token '{TARGET_LANG_CODE}' into tokenizer vocabulary...")
num_added = tokenizer.add_special_tokens({"additional_special_tokens": [TARGET_LANG_CODE]})
print(f"[✓] Vocabulary expanded. New size: {len(tokenizer)}")

# Configure 4-bit quantization
# If you are not GPU poor you can skip quantization, this can lead to a better loss
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("[*] Loading NLLB-200 model with 4-bit quantization...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    NLLB_MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)

# Resize model token embeddings and clone anchor language weights 
print(f"[*] Resizing embedding matrix to {len(tokenizer)} and cloning weights from {SIMILAR_LANG_CODE}...")
model.resize_token_embeddings(len(tokenizer))

new_token_id = tokenizer.convert_tokens_to_ids(TARGET_LANG_CODE)
similar_token_id = tokenizer.convert_tokens_to_ids(SIMILAR_LANG_CODE)

with torch.no_grad():
    model.model.shared.weight[new_token_id] = model.model.shared.weight[similar_token_id].clone()
    if hasattr(model, "lm_head") and model.lm_head.weight.shape[0] == len(tokenizer):
        model.lm_head.weight[new_token_id] = model.lm_head.weight[similar_token_id].clone()

model = prepare_model_for_kbit_training(model)

# Tokenization function mapping
def tokenize_nllb_fn(examples):
    inputs = examples[SRC_COL]
    targets = examples[TGT_COL]
    
    tokenizer.src_lang = "eng_Latn"
    tokenizer.tgt_lang = TARGET_LANG_CODE

    model_inputs = tokenizer(
        inputs,
        text_target=targets,
        max_length=512,
        truncation=True,
        padding=False
    )
    return model_inputs

tokenized_datasets = dataset_package.map(tokenize_nllb_fn, batched=True, remove_columns=[SRC_COL, TGT_COL])

# Inject PEFT LoRA config
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "embed_tokens", "lm_head"]  # Targets embedding + linear layers
)

model = get_peft_model(model, peft_config)

model.is_model_parallel = True
model.model_parallel = True

model.print_trainable_parameters()

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=f"/kaggle/working/nllb_peft_{TARGET_LANG_CODE.lower()}",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4, # You can chose your own learning rate & scheduler  
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=7, # Increase number of epochs if you have enough VRAM
    predict_with_generate=True,
    fp16=True,
    logging_steps=50,
    report_to="none",
    push_to_hub=True,
    hub_model_id=FULL_REPO_ID,
    hub_strategy="every_save",
    hub_private_repo=True
)

# Prevent trainer from wrapping model in data parallel on dual GPUs
training_args._n_gpu = 1

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

trainer.is_model_parallel = True

print(f"[*] Starting fine-tuning for [{TARGET_LANG_CODE}] ...")
trainer.train()

# Push adapter weights and expanded tokenizer
print(f"[*] Pushing fine-tuned adapters to {FULL_REPO_ID}...")
trainer.push_to_hub(commit_message="Training complete")
tokenizer.push_to_hub(FULL_REPO_ID)
print("[✓] Model and tokenizer successfully pushed to Hugging Face Hub")

### Evaluation

In [ ]:
!pip install -q evaluate sacrebleu

import evaluate
import sacrebleu
import torch
import pandas as pd

print("[*] Launching NLLB evaluator...")

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

eval_subset = dataset_package["test"].select(range(min(len(dataset_package["test"]), 200)))

references = []
predictions = []

model.eval()
target_lang_id = tokenizer.convert_tokens_to_ids(TARGET_LANG_CODE)

for item in eval_subset:
    tokenizer.src_lang = "eng_Latn"
    inputs = tokenizer(item[SRC_COL], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=target_lang_id,
            max_length=512
        )
    decoded_pred = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
    predictions.append(decoded_pred)
    references.append(item[TGT_COL])

bleu_results = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
chrf_results = chrf_metric.compute(predictions=predictions, references=[[r] for r in references])

print(f"\n================ FINAL PERFORMANCE REPORT [{TARGET_LANG_CODE}] ================")
print(f"| -> SacreBLEU Score  : {bleu_results['score']:.3f}")
print(f"| -> chrF++ Score     : {chrf_results['score']:.3f}")
print("==============================================================================\n")

In [ ]:
import pandas as pd

# Comparison table
original_english = [item[SRC_COL] for item in eval_subset]

analysis_df = pd.DataFrame({
    "English (Source)": original_english,
    f"Ground Truth ({TARGET_LANG_CODE})": references,
    "Adapter Output (Prediction)": predictions
})

analysis_df['Perfect Match'] = analysis_df[f"Ground Truth ({TARGET_LANG_CODE})"].str.strip() == analysis_df['Adapter Output (Prediction)'].str.strip()

print(f"Sample predictions (Perfect Matches: {analysis_df['Perfect Match'].sum()}/{len(analysis_df)}):")
display(analysis_df.head(20))

In [ ]:
# Test with unseen English sentencs

def translate_new_sentence(english_text, model, tokenizer, target_lang=TARGET_LANG_CODE):
    """
    Translates an unseen English sentence into local language.
    """
    model.eval()
    tokenizer.src_lang = "eng_Latn"

    inputs = tokenizer(english_text, return_tensors="pt").to(model.device)
    target_lang_id = tokenizer.convert_tokens_to_ids(target_lang)

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=target_lang_id,
            max_length=512,
            num_beams=5,
            early_stopping=True
        )

    translated_text = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
    return translated_text

print(f"Interactive translation [{TARGET_LANG_CODE}] ---")
print("Type 'quit' or 'exit' to stop.\n")

while True:
    user_sentence = input("Enter an English sentence to translate: ")

    if user_sentence.lower().strip() in ['quit', 'exit']:
        print("Playground closed.")
        break

    if not user_sentence.strip():
        print("Please enter a valid sentence.")
        print("-" * 40)
        continue

    translation = translate_new_sentence(user_sentence, model, tokenizer, target_lang=TARGET_LANG_CODE)
    print(f"Translation ({TARGET_LANG_CODE}): {translation}")
    print("-" * 40)